# Cross-Regional Events Co-occurrence: Full Dataset

Scope (meeting with Yixiao, Sept 2026):

- Pairs PL-RSC, PL-HPC, RSC-HPC; delta and spindle
- Non-zero lag only, no iCOH
- Wave similarity: Pearson + cross-correlation using each event's own start/end (my choice, not confirmed)
- No claim about which region leads
- Over time: per-rat Spearman + permutation only, no Wilcoxon / FDR

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# lab repo root on the path, so `modules` imports work from this folder
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "modules" / "project_config.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from modules.project_config import get_path  # noqa: E402

# local paths come from config.yaml (see config.example.yaml)
ROOT = get_path("R1_8_root")  # holds Rat_HM_Ephys_TD_Analysis_R1_8 and _R9_16
assert get_path("R9_16_root") == ROOT, "these notebooks expect R1_8_root == R9_16_root"
WAVE_CACHE = get_path("event_cache_root")
SESSION_MAP_CSV = get_path("session_map_csv")
DAY_GRID_XLSX = get_path("day_grid_xlsx")
OUT = REPO_ROOT / "results" / "leticia_event_coupling"  # figures and caches
SPINDLE_METHOD = "wavelet_amp_1_ampcore_3"
PHASES = ["postsleep"]

BOUT_KEYS = ["rat", "date", "trial", "nrem_bout_index"]
AREAS = ["PL", "RSC", "HPC"]
RATS = [1, 3, 7, 8, 11, 12, 13]

N_ITER = 1000
SHIFT_RATS = RATS

FIGDIR = OUT / "cross_regional_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(FIGDIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved:", name)


print("drive reachable:", ROOT.exists(), " (False is fine - we load from WAVE_CACHE)")
print("rats:", RATS, "| circular shift on:", SHIFT_RATS, "|", N_ITER, "iterations")

## Load

Bad days are removed in `build_wave` (`excluded_days.py`).

In [ ]:
from modules.event_coupling.build_tables import add_condition, build_wave

ripples = add_condition(
    build_wave(ROOT, "ripple", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
deltas = add_condition(
    build_wave(ROOT, "delta", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
spindles = add_condition(
    build_wave(ROOT, "spindle", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

rip = ripples[(ripples["rat"].isin(RATS)) & (ripples["group"] != "other")]
dlt = deltas[
    (deltas["rat"].isin(RATS))
    & (deltas["group"] != "other")
    & (deltas["area"].isin(AREAS))
]
spn = spindles[
    (spindles["rat"].isin(RATS))
    & (spindles["group"] != "other")
    & (spindles["area"].isin(AREAS))
]

print(f"{len(rip):,} ripples | {len(dlt):,} deltas | {len(spn):,} spindles")
print(rip.groupby("group").size().to_string())

### Check that the bad days were dropped

In [ ]:
from modules.event_coupling.excluded_days import excluded_table

ex = excluded_table()

for name, df in [("ripples", rip), ("deltas", dlt), ("spindles", spn)]:
    check = ex.merge(
        df[["rat", "date"]]
        .assign(date=pd.to_datetime(df["date"]).dt.normalize())
        .drop_duplicates(),
        on=["rat", "date"],
    )
    print(
        f"{name}: {len(ex)} rat-days on the exclusion list | "
        f"still present: {len(check)} (want 0)"
    )
    assert len(check) == 0, (
        f"exclusion did not fire for {name} - is excluded_days.py in this folder?"
    )

## Time labels

In [ ]:
from modules.event_coupling.gl_sessions import add_gl_sessions, build_gl_sessions
from modules.event_coupling.hc_sessions import add_session_type, build_day_table


def prep_keys(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["rat"] = df["rat"].astype("int64")
    return df


gl = prep_keys(build_gl_sessions(ROOT, cache_dir=WAVE_CACHE, rebuild=False))
days = add_session_type(build_day_table(DAY_GRID_XLSX, verbose=False))


def add_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    return df.merge(
        prep_keys(days)[["rat", "date", "session_type"]].drop_duplicates(),
        on=["rat", "date"],
        how="left",
    )


rip_ctx = add_context(rip)
dlt_ctx = add_context(dlt)

# days since each rat's first recording (gaps between sessions are 1-7 days)
first = rip_ctx.groupby("rat")["date"].transform("min")
rip_ctx["days_elapsed"] = (rip_ctx["date"] - first).dt.days
dlt_ctx = dlt_ctx.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

print(
    rip_ctx.groupby(["rat", "group"])["date"]
    .nunique()
    .unstack(fill_value=0)
    .to_string()
)

## Build spindle context

Adds GL/session/days_elapsed labels to `spn`.

In [ ]:
def add_spindle_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    df = df.merge(
        prep_keys(days)[["rat", "date", "session_type"]].drop_duplicates(),
        on=["rat", "date"],
        how="left",
    )
    return df


spn_ctx = add_spindle_context(spn)
spn_ctx = spn_ctx.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

PAIRS = [("PL", "RSC"), ("PL", "HPC"), ("RSC", "HPC")]
print(f"{len(spn_ctx):,} spindle rows with context")
print("pairs:", PAIRS)

## Self-tests

Synthetic data with a known answer.

In [ ]:
def count_events_with_partner(seed, target, lag_lo, lag_hi, seed_col, target_col):
    """Unique seed-region events with a target-region event in the given lag
    window, per rat-day. Generalized version of the ripple-delta partner
    counter used elsewhere in this project -- works for any same-type event
    pair across two regions (e.g. delta_PL seed, delta_RSC target).
    lag = seed_peak - target_peak. Matches only within a trial.
    """
    rows = []
    for (rat, date, trial), sg in seed.groupby(["rat", "date", "trial"]):
        tg = target[
            (target["rat"] == rat)
            & (target["date"] == date)
            & (target["trial"] == trial)
        ]
        n_coupled = 0
        for bout in sg["nrem_bout_index"].unique():
            s_b = sg.loc[sg["nrem_bout_index"] == bout, seed_col].values
            t_b = (
                tg.loc[tg["nrem_bout_index"] == bout, target_col].values
                if not tg.empty
                else np.array([])
            )
            if len(s_b) == 0 or len(t_b) == 0:
                continue
            lags = s_b[:, None] - t_b[None, :]
            has_partner = ((lags >= lag_lo) & (lags < lag_hi)).any(axis=1)
            n_coupled += int(has_partner.sum())
        rows.append(
            {
                "rat": rat,
                "date": date,
                "trial": trial,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
            }
        )
    return pd.DataFrame(rows)


def _synth_cross_regional(n_days=2, seed=0):
    rng = np.random.default_rng(seed)
    A, B = [], []
    for day in range(1, n_days + 1):
        a = np.sort(rng.uniform(0, 20000, 40))
        b = np.sort(rng.uniform(0, 20000, 40))
        if day == 2:
            take = rng.choice(len(a), 6, replace=False)
            b = np.sort(np.concatenate([b, a[take] + 80]))
        b = b[(b > 0) & (b < 20000)]
        A.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "trial": 1,
                    "nrem_bout_index": 0,
                    "peak_a": a,
                }
            )
        )
        B.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "trial": 1,
                    "nrem_bout_index": 0,
                    "peak_b": b,
                }
            )
        )
    return pd.concat(A, ignore_index=True), pd.concat(B, ignore_index=True)


_seedA, _targetB = _synth_cross_regional()
_res = count_events_with_partner(_seedA, _targetB, -150, -20, "peak_a", "peak_b")
assert _res.loc[_res.date == "day2", "n_coupled"].iloc[0] >= 6
assert (
    _res.loc[_res.date == "day2", "n_coupled"].iloc[0]
    > _res.loc[_res.date == "day1", "n_coupled"].iloc[0]
)
print("Cross-regional partner-counting self-test PASSED")

In [ ]:
from scipy.stats import pearsonr


def wave_similarity(wave_a, wave_b, max_lag_frac=0.5):
    """Pearson correlation (after resampling both to a common length) and
    cross-correlation (best lag + peak correlation), between two waveforms
    of possibly different lengths -- each waveform is taken from its own
    event's detected start/end window, so lengths naturally differ.
    """
    n = 100
    a_rs = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(wave_a)), wave_a)
    b_rs = np.interp(np.linspace(0, 1, n), np.linspace(0, 1, len(wave_b)), wave_b)

    a_z = (a_rs - a_rs.mean()) / (a_rs.std() + 1e-12)
    b_z = (b_rs - b_rs.mean()) / (b_rs.std() + 1e-12)

    pearson_r, _ = pearsonr(a_rs, b_rs)

    xcorr = np.correlate(a_z, b_z, mode="full") / n
    lags = np.arange(-n + 1, n)
    max_lag = int(n * max_lag_frac)
    mask = np.abs(lags) <= max_lag
    xcorr, lags = xcorr[mask], lags[mask]
    best_idx = np.argmax(np.abs(xcorr))

    return {
        "pearson_r": pearson_r,
        "xcorr_peak": xcorr[best_idx],
        "xcorr_best_lag": lags[best_idx],
    }


_t = np.linspace(0, 2 * np.pi, 80)
_wave = np.sin(_t)
_res_identical = wave_similarity(_wave, _wave.copy())
assert _res_identical["pearson_r"] > 0.99

_rng = np.random.default_rng(0)
_noise = _rng.normal(0, 1, 80)
_res_noise = wave_similarity(_wave, _noise)
assert abs(_res_noise["pearson_r"]) < 0.3

_r1 = _res_identical["pearson_r"]
_r2 = _res_noise["pearson_r"]
print(f"Wave-similarity self-test PASSED (identical r={_r1:.2f}, noise r={_r2:.2f})")

## [SILENCED] Circular-shift method (all-pairs counting)

Original all-pairs method with a circular-shift surrogate. Not used: it gave a "mound" artifact from shared rate co-modulation (r = 0.54 between PL and RSC spindle rate).

Kept commented out in case the nearest-neighbour method needs a significance test later.

Groups by `(rat, date)` without `trial`, add `trial` before using it again.

In [ ]:
# import pickle

# CACHE_XREGIONAL = OUT / "significance_cache_crossregional"; CACHE_XREGIONAL.mkdir(parents=True, exist_ok=True)
# N_ITER_SIG = 1000
# CHECKPOINT_EVERY = 50   # was 200; delta-delta pairs are much slower per
#                         # iteration than ripple-delta (see below), so losing
#                         # 200 iterations in a crash would cost a lot more time
# edges = np.arange(-1000, 1020, 20)
# left = edges[:-1]


# def windowed_hist(s_b, t_b, lo, hi, edges):
#     """Vectorized histogram of (seed - target) lags, only for pairs within
#     [lo, hi], using searchsorted + a ragged-range trick instead of a full
#     outer product. Checked on synthetic data: identical results to the simple
#     np.histogram(s[:,None]-t) version, 40-55x faster for dense event trains.

#     Needed here because, unlike ripple-delta (one sparse and one dense train),
#     cross-regional delta-delta and spindle-spindle compare two dense trains,
#     and the simple full pairwise version was taking hours.
#     """
#     lo_idx = np.searchsorted(t_b, s_b + lo, side="left")
#     hi_idx = np.searchsorted(t_b, s_b + hi, side="right")
#     counts = hi_idx - lo_idx
#     total = counts.sum()
#     if total == 0:
#         return np.zeros(len(edges) - 1)
#     seed_idx_repeated = np.repeat(np.arange(len(s_b)), counts)
#     group_offsets = np.repeat(np.cumsum(counts) - counts, counts)
#     offset_within_group = np.arange(total) - group_offsets
#     j_indices = np.repeat(lo_idx, counts) + offset_within_group
#     lags_flat = s_b[seed_idx_repeated] - t_b[j_indices]
#     return np.histogram(lags_flat, bins=edges)[0]


# def cross_regional_dist(seed_df, target_df, area_seed, area_target, wave,
#                         n_iter=N_ITER_SIG, seed=0, checkpoint_every=CHECKPOINT_EVERY,
#                         verbose=True):
#     """1000-iteration circular-shift surrogate for one region pair and one
#     wave type. Saves a checkpoint every `checkpoint_every` iterations, so an
#     interruption only loses up to that many iterations.

#     Keeps the full (n_iter, n_bins) null array, same exact-count method as
#     the ripple-delta/spindle-ripple surrogates in the rest of the project.

#     Resuming: rng.bit_generator.state is saved and restored, so the result
#     is identical to a run that wasn't interrupted (checked on synthetic
#     data before).

#     Which region is "seed" and which is "target" doesn't matter (direction
#     isn't tested); only the seed region's train is shifted.
#     """
#     key = f"{wave}_{area_seed}_{area_target}_n{n_iter}_seed{seed}"
#     final_f = CACHE_XREGIONAL / f"{key}.pkl"
#     ckpt_f = CACHE_XREGIONAL / f"{key}_partial.pkl"

#     if final_f.exists():
#         if verbose:
#             print(f"  {area_seed}-{area_target} {wave}: loaded from cache")
#         return pickle.load(open(final_f, "rb"))

#     peak_col = f"{wave}_peak_index"
#     s = seed_df[seed_df["area"] == area_seed]
#     t = target_df[target_df["area"] == area_target]
#     groups = list(s.groupby(["rat", "date"]))
#     lo_edge, hi_edge = edges[0], edges[-1]

#     real = np.zeros(len(edges) - 1)
#     for (rat, date), sg in groups:
#         tg = t[(t["rat"] == rat) & (t["date"] == date)]
#         for bout in sg["nrem_bout_index"].unique():
#             s_b = np.sort(sg.loc[sg["nrem_bout_index"] == bout, peak_col].values)
#             t_b = np.sort(tg.loc[tg["nrem_bout_index"] == bout, peak_col].values) if not tg.empty else np.array([])
#             if len(s_b) == 0 or len(t_b) == 0:
#                 continue
#             real += windowed_hist(s_b, t_b, lo_edge, hi_edge, edges)

#     null = np.zeros((n_iter, len(edges) - 1))
#     rng = np.random.default_rng(seed)
#     start_i = 0

#     if ckpt_f.exists():
#         ck = pickle.load(open(ckpt_f, "rb"))
#         start_i = ck["n_done"]
#         null[:start_i] = ck["null_partial"]
#         rng.bit_generator.state = ck["rng_state"]
#         if verbose:
#             print(f"  {area_seed}-{area_target} {wave}: resuming from {start_i}/{n_iter}")

#     for i in range(start_i, n_iter):
#         iter_hist = np.zeros(len(edges) - 1)
#         for (rat, date), sg in groups:
#             tg = t[(t["rat"] == rat) & (t["date"] == date)]
#             for bout in sg["nrem_bout_index"].unique():
#                 s_b = np.sort(sg.loc[sg["nrem_bout_index"] == bout, peak_col].values)
#                 t_b = np.sort(tg.loc[tg["nrem_bout_index"] == bout, peak_col].values) if not tg.empty else np.array([])
#                 if len(s_b) == 0 or len(t_b) == 0:
#                     continue
#                 span = (s_b.max() - s_b.min()) if len(s_b) > 1 else 5000
#                 shift = rng.uniform(1000, max(span, 1001))
#                 s_shifted = np.sort((s_b - s_b.min() + shift) % (span + 1) + s_b.min())
#                 iter_hist += windowed_hist(s_shifted, t_b, lo_edge, hi_edge, edges)
#         null[i] = iter_hist

#         if (i + 1) % checkpoint_every == 0 or (i + 1) == n_iter:
#             pickle.dump({"n_done": i + 1, "null_partial": null[:i + 1],
#                         "rng_state": rng.bit_generator.state},
#                        open(ckpt_f, "wb"))
#             if verbose:
#                 print(f"    [checkpoint: {i + 1}/{n_iter}]")

#     dist = {"real": real, "null": null,
#            "z": (real - null.mean(axis=0)) / (null.std(axis=0) + 1e-9)}
#     pickle.dump(dist, open(final_f, "wb"))
#     if ckpt_f.exists():
#         ckpt_f.unlink()
#     if verbose:
#         print(f"  {area_seed}-{area_target} {wave}: computed and cached")
#     return dist


# def peak_latency_ms(dist):
#     return int(left[np.argmax(dist["real"])])


# def window_significance_xr(dist, window_ms):
#     """Exact-count p-value: how many of the n_iter shuffles were >= the
#     observed value, same method as in the rest of the project, no
#     distributional assumption."""
#     lo, hi = window_ms
#     win = (left >= lo) & (left < hi)
#     observed = dist["real"][win].sum()
#     null_sums = dist["null"][:, win].sum(axis=1)
#     p = (np.sum(null_sums >= observed) + 1) / (len(null_sums) + 1)
#     return {"observed": observed, "null_mean": null_sums.mean(), "null_sd": null_sums.std(),
#            "ratio": observed / null_sums.mean() if null_sums.mean() > 0 else np.nan,
#            "p": p, "significant": p < 0.05}


# print("cross-regional surrogate + significance functions defined "
#      "(vectorized windowed histogram, checkpointed every 50, exact-count p-values)")

In [ ]:
# def plot_cross_regional(dist, area_seed, area_target, wave, window_ms):
#     """Absolute-count histogram, null +/-2SD band, non-zero-lag peak marked."""
#     mu, sd = dist["null"].mean(axis=0), dist["null"].std(axis=0)
#     peak_ms = peak_latency_ms(dist)
#     res = window_significance_xr(dist, window_ms)
#     ratio_val = res["ratio"]
#     p_val = res["p"]

#     fig, ax = plt.subplots(figsize=(9, 4.5))
#     ax.bar(left, dist["real"], width=20, align="edge", color="0.35", zorder=2)
#     ax.plot(left, mu, color="0.5", lw=1, zorder=3, label="null mean")
#     ax.fill_between(left, mu - 2*sd, mu + 2*sd, color="tab:orange", alpha=0.25,
#                     zorder=1, label="null \u00b12 SD")
#     ax.axvline(0, color="k", lw=1)
#     ax.axvline(peak_ms, color="tab:red", lw=1.5, ls="--",
#               label=f"peak: {peak_ms} ms")
#     ax.set_xlabel(f"{area_seed} {wave} peak \u2212 {area_target} {wave} peak (ms)")
#     ax.set_ylabel("count")
#     ax.set_title(f"{area_seed}\u2013{area_target}  {wave}   "
#                  f"peak {peak_ms} ms, ratio {ratio_val:.2f}x, p={p_val:.3f}")
#     ax.legend(fontsize=8)
#     plt.tight_layout()
#     save_fig(fig, f"xregional_{wave}_{area_seed}_{area_target}")
#     plt.show()

#     zero_mask = np.abs(left) < 20
#     zero_lag_deficit = dist["real"][zero_mask].mean() < mu[zero_mask].mean()
#     direction_note = "non-zero -- consistent with real coupling, not volume conduction" if peak_ms != 0 else "AT ZERO -- check for volume conduction"
#     print(f"{area_seed}\u2013{area_target} {wave}: peak {peak_ms} ms ({direction_note})")
#     if zero_lag_deficit:
#         print("  additionally: a deficit (not excess) at zero lag -- the opposite of what volume conduction would produce")
#     sig_val = res["significant"]
#     print(f"  ratio {ratio_val:.2f}x | p={p_val:.4f} | significant={sig_val}")
#     return res, peak_ms


# WINDOW_MS = (-250, 250)   # symmetric window; direction is not being tested here

# xregional_results = []
# for area_seed, area_target in PAIRS:
#     for wave, seed_ctx, target_ctx in [("delta", dlt_ctx, dlt_ctx), ("spindle", spn_ctx, spn_ctx)]:
#         print(f"=== {area_seed}\u2013{area_target}, {wave} ===")
#         dist = cross_regional_dist(seed_ctx, target_ctx, area_seed, area_target, wave)
#         res, peak_ms = plot_cross_regional(dist, area_seed, area_target, wave, WINDOW_MS)
#         xregional_results.append({"pair": f"{area_seed}-{area_target}", "wave": wave,
#                                   "peak_ms": peak_ms, **res})
#         print()

# xregional_summary = pd.DataFrame(xregional_results)
# print(xregional_summary.to_string(index=False))

## Cross-regional co-occurrence: nearest-neighbor method

Nearest-neighbour method (Yixiao, after Schreiner et al. 2022): for each seed event, the nearest target event within +/-750 ms.

Each seed event gives at most one match, so no mound.

Exploratory, no significance test yet. Peak checked with 5/10/20/50 ms bins.

In [ ]:
def nearest_within_window(seed_df, target_df, area_seed, area_target, wave, window_ms):
    """For each seed-region event, the single nearest target-region event
    within window_ms (if any). Returns an array of peak differences
    (seed - target).

    Matching happens within one recording (rat, date, TRIAL) and one NREM bout.
    Event indices restart at 0 in every trial, and so do bout indices, so
    matching on (rat, date, bout) alone pairs events from different trials --
    the bug this replaced."""
    peak_col = f"{wave}_peak_index"
    s = seed_df[seed_df["area"] == area_seed]
    t = target_df[target_df["area"] == area_target]
    diffs = []
    for (rat, date, trial), sg in s.groupby(["rat", "date", "trial"]):
        tg = t[(t["rat"] == rat) & (t["date"] == date) & (t["trial"] == trial)]
        for bout in sg["nrem_bout_index"].unique():
            s_b = sg.loc[sg["nrem_bout_index"] == bout, peak_col].values
            t_b = (
                tg.loc[tg["nrem_bout_index"] == bout, peak_col].values
                if not tg.empty
                else np.array([])
            )
            if len(s_b) == 0 or len(t_b) == 0:
                continue
            for p in s_b:
                candidates = t_b[np.abs(t_b - p) <= window_ms]
                if len(candidates) == 0:
                    continue
                nearest = candidates[np.argmin(np.abs(candidates - p))]
                diffs.append(p - nearest)
    return np.array(diffs)


def _synth_nn_test(true_lag=80, n_events=200, seed=0):
    """Self-test: plant a known lag, confirm nearest-neighbor recovers it."""
    rng = np.random.default_rng(seed)
    s_b = np.sort(rng.uniform(0, 60000, n_events))
    t_b = np.sort(rng.uniform(0, 60000, n_events))
    take = rng.choice(len(s_b), int(0.3 * len(s_b)), replace=False)
    extra = s_b[take] + true_lag
    t_b = np.sort(np.concatenate([t_b, extra[(extra > 0) & (extra < 60000)]]))
    return s_b, t_b


_s_test, _t_test = _synth_nn_test()
_test_df_s = pd.DataFrame(
    {
        "rat": 1,
        "date": "d1",
        "area": "X",
        "trial": 1,
        "nrem_bout_index": 0,
        "wave_peak_index": _s_test,
    }
)
_test_df_t = pd.DataFrame(
    {
        "rat": 1,
        "date": "d1",
        "area": "Y",
        "trial": 1,
        "nrem_bout_index": 0,
        "wave_peak_index": _t_test,
    }
)
_diffs_test = nearest_within_window(_test_df_s, _test_df_t, "X", "Y", "wave", 750)
_peak_test = np.arange(-750, 770, 20)[
    np.argmax(np.histogram(_diffs_test, bins=np.arange(-750, 770, 20))[0])
]
assert abs(_peak_test - (-80)) <= 40 or abs(_peak_test - 80) <= 40, (
    f"nearest-neighbor self-test FAILED: expected peak near +/-80ms, got {_peak_test}"
)
print(
    f"nearest-neighbor self-test PASSED (planted lag 80ms, detected peak near {_peak_test}ms)"
)

# events from different trials never match
_x = pd.DataFrame(
    {
        "rat": 1,
        "date": "d1",
        "area": "X",
        "trial": 1,
        "nrem_bout_index": 0,
        "wave_peak_index": [1000],
    }
)
_y_other = pd.DataFrame(
    {
        "rat": 1,
        "date": "d1",
        "area": "Y",
        "trial": 2,
        "nrem_bout_index": 0,
        "wave_peak_index": [1050],
    }
)
assert len(nearest_within_window(_x, _y_other, "X", "Y", "wave", 750)) == 0, (
    "matched across trials"
)
assert list(
    nearest_within_window(_x, _y_other.assign(trial=1), "X", "Y", "wave", 750)
) == [-50]
print("trial-separation self-test PASSED (no matches across trials)")

### Run for all pairs and both wave types

Bin width differs between delta and spindle only for plotting.

In [ ]:
BIN_WIDTH = {"delta": 10, "spindle": 50}

xregional_diffs = {}  # keep for the significance-testing step later

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for row, wave in enumerate(["delta", "spindle"]):
    ctx = dlt_ctx if wave == "delta" else spn_ctx
    bw = BIN_WIDTH[wave]
    for col, (area_seed, area_target) in enumerate(PAIRS):
        diffs = nearest_within_window(ctx, ctx, area_seed, area_target, wave, 750)
        xregional_diffs[(area_seed, area_target, wave)] = diffs

        ax = axes[row, col]
        edges = np.arange(-750, 750 + bw, bw)
        ax.hist(
            diffs,
            bins=edges,
            alpha=0.7,
            color="tab:blue" if wave == "delta" else "tab:orange",
        )
        ax.axvline(0, color="k", lw=1)
        ax.set_title(f"{area_seed}\u2013{area_target}  {wave}")
        ax.set_xlabel("peak difference (ms)")

        peaks = []
        for check_bw in [5, 10, 20, 50]:
            e = np.arange(-750, 750 + check_bw, check_bw)
            h, _ = np.histogram(diffs, bins=e)
            peaks.append(e[np.argmax(h)] + check_bw / 2)
        print(
            f"{area_seed}-{area_target} {wave}: peak across bin widths [5,10,20,50ms] = "
            f"{[round(p, 1) for p in peaks]} | median={np.median(diffs):.1f}ms | n={len(diffs):,}"
        )

plt.tight_layout()
plt.show()

## Wave similarity for co-occurring pairs

Pearson and cross-correlation between paired waveforms (Yixiao). Waveforms cut from the band-passed trace (delta 0.5-4 Hz, spindle 9-20 Hz) using each event's own start/end.

- Pairs: nearest target event in the same recording and bout, within +/-200 ms (delta) or +/-750 ms (spindle)
- Sample: 8% of recordings, stratified by rat
- Control: same seed waveform with a non-co-occurring target from the same bout; use paired minus control
- Delta: Pearson r. Spindle: |xcorr_peak|

In [ ]:
import io
import shutil
import time as _time

import scipy.io
from scipy.signal import butter, sosfiltfilt

FS = 1000  # 1000 Hz, as in PROJECT_DATA_REFERENCE.md
BAND = {"delta": [0.5, 4], "spindle": [9, 20]}  # Yixiao, Sept 2026

PREPROCESSED_BASE = ROOT  # Z:\Rat\HM\Rat_HM_Ephys_TD, checked with two real
# examples (rat 13, and the earlier rat-1-8 screenshot)


def cohort_paths(rat):
    """Map a rat number to its cohort folder pair, e.g. rat 13 ->
    ("R9_16", "R13-16") -- validated against a confirmed real path from
    Yixiao/the drive before use."""
    top = "R1_8" if rat <= 8 else "R9_16"
    if rat <= 4:
        sub = "R1-4"
    elif rat <= 8:
        sub = "R5-8"
    elif rat <= 12:
        sub = "R9-12"
    else:
        sub = "R13-16"
    return top, sub


def filter_lfp(lfp, fs, band):
    """Zero-phase band-pass filter, per Yixiao (Sept 2026). Validated on
    synthetic mixed-frequency data before use: correctly isolates each
    target band (r>0.99 vs the true component, r~0 vs other components)."""
    sos = butter(N=4, Wn=band, btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, lfp)


_mat_cache = {}  # avoid re-loading the same multi-MB .mat file for every event


class UnreadableMatFile(FileNotFoundError):
    """The .mat file exists but could not be read (network / drive problem).
    A subclass of FileNotFoundError, so code that skips missing files also
    skips this instead of crashing a long run."""


def _load_mat_signal(
    path, chunk_bytes=8 * 1024 * 1024, attempts=6, waits=(10, 20, 30, 40, 60)
):
    """Load the 'data' array of one .mat file from the (network) drive.

    Reading straight off the Z: drive with scipy.io.loadmat(path) failed with
    OSError [Errno 22 Invalid argument] part-way through a file. The cause is
    not established, so this is a mitigation: the file is first copied into
    memory in 8 MB pieces (small, predictable reads) and parsed from memory,
    and a failed read is retried in case it is a transient network error. If
    it still fails, UnreadableMatFile is raised so the caller can skip this
    recording and say which file it was."""
    last = None
    for attempt in range(1, attempts + 1):
        try:
            buf = io.BytesIO()
            with open(path, "rb") as f:
                shutil.copyfileobj(f, buf, chunk_bytes)
            buf.seek(0)
            return scipy.io.loadmat(buf)["data"].squeeze()
        except OSError as e:
            last = e
            if attempt < attempts:
                _time.sleep(waits[min(attempt - 1, len(waits) - 1)])
    raise UnreadableMatFile(f"{path} unreadable after {attempts} attempts: {last}")


def load_channel_signal(rat, date, area, channel, trial, phase="postsleep"):
    """Load one channel's continuous PreprocessedData signal for one
    rat/date/trial. Cached per key since one .mat file covers every event
    within that trial.

    Two filename formats exist: most days are split per trial
    (chan{channel}_{trial}.mat); twelve early Sept 2022 recordings (rats
    1-8) are one continuous unsplit file per channel (chan{channel}.mat, no
    trial suffix). Tries the per-trial filename first, falls back to the
    unsplit filename.

    .mat variable name confirmed: dict_keys(['__header__', '__version__',
    '__globals__', 'data']) -- the signal itself is under 'data'.
    """
    key = (rat, date, area, channel, trial, phase)
    if key in _mat_cache:
        return _mat_cache[key]

    top, sub = cohort_paths(rat)
    date_str = pd.to_datetime(date).strftime("%Y%m%d")
    day_dir = (
        PREPROCESSED_BASE
        / f"Rat_HM_Ephys_TD_Analysis_{top}"
        / sub
        / "PreprocessedData"
        / area
        / str(rat)
        / date_str
        / phase
    )

    trial_path = day_dir / f"chan{channel}_{trial}.mat"
    unsplit_path = day_dir / f"chan{channel}.mat"
    path = trial_path if trial_path.exists() else unsplit_path
    if not path.exists():
        raise FileNotFoundError(
            f"neither {trial_path.name} nor {unsplit_path.name} found in {day_dir}"
        )

    signal = _load_mat_signal(path)  # chunked read + retry (see helper above)
    _mat_cache[key] = signal
    return signal


def get_waveform(row, wave):
    """Extract one event's filtered waveform, using its own detected
    start/end indices, from the raw PreprocessedData signal.

    NOT used by the sampled wave-similarity cell below: this filters each
    short slice separately, which is not equivalent to slicing a filtered
    continuous trace (synthetic check: median r 0.72, 10th percentile ~0).
    Kept only for looking at a single event."""
    signal = load_channel_signal(
        row["rat"], row["date"], row["area"], row["channel"], row["trial"]
    )
    start, end = int(row[f"{wave}_start_index"]), int(row[f"{wave}_end_index"])
    raw_segment = signal[start:end]
    return filter_lfp(raw_segment, FS, BAND[wave])

In [ ]:
CACHE_WAVE_SIM = OUT / "wave_similarity_cache_files"
CACHE_WAVE_SIM.mkdir(parents=True, exist_ok=True)
print("CACHE_WAVE_SIM path:", CACHE_WAVE_SIM.resolve())
print("dlt_ctx max delta_peak_index:", f"{dlt_ctx['delta_peak_index'].max():,.0f}")
print("dlt_ctx total rows:", f"{len(dlt_ctx):,}")
print("spn_ctx max spindle_peak_index:", f"{spn_ctx['spindle_peak_index'].max():,.0f}")
print("Is this the FULL dataset?", dlt_ctx["delta_peak_index"].max() > 3_600_000)

In [ ]:
import pickle
import time

CACHE_WAVE_SIM = OUT / "wave_similarity_cache_files"
CACHE_WAVE_SIM.mkdir(parents=True, exist_ok=True)
FILE_FRACTION = 0.08  # fraction of (rat, date, trial) recordings sampled, per wave type
SAMPLE_SEED = 0
WAVE_HALF_WINDOW_MS = {"delta": 200, "spindle": 750}  # a seed event pairs with the
# nearest target event within +/- this


def sample_units(ctx, fraction=FILE_FRACTION, seed=SAMPLE_SEED):
    """Random sample of recordings -- (rat, date, trial) units -- stratified
    by rat, so every rat is represented and the sample is a fixed fraction
    of each rat's recordings. This mirrors the bout-level 8% validation in
    PROJECT_DATA_REFERENCE.md (sample whole recordings, use everything inside
    them), rather than sampling individual pairs scattered across files."""
    units = (
        ctx[["rat", "date", "trial"]]
        .drop_duplicates()
        .sort_values(["rat", "date", "trial"])
        .reset_index(drop=True)
    )
    rng = np.random.default_rng(seed)
    picked = []
    for rat, g in units.groupby("rat"):
        n = max(int(round(fraction * len(g))), 1)
        picked.append(g.iloc[np.sort(rng.choice(len(g), size=n, replace=False))])
    return pd.concat(picked).reset_index(drop=True)


_filtered_cache = {}


def filtered_signal(rat, date, area, channel, trial, wave):
    """The WHOLE continuous trace, band-pass filtered once, then sliced per
    event. Filtering each ~400 ms event slice separately is not equivalent
    (for delta the two disagreed badly in a synthetic check: median r 0.72,
    10th percentile ~0), because a 0.5 Hz filter needs seconds of signal."""
    key = (rat, date, area, channel, trial, wave)
    if key not in _filtered_cache:
        raw_sig = load_channel_signal(rat, date, area, channel, trial)
        _filtered_cache[key] = filter_lfp(raw_sig, FS, BAND[wave])
    return _filtered_cache[key]


def match_pairs_in_unit(sg, tg, peak_col, half_window, rng):
    """Within ONE recording (rat, date, trial), and within each NREM bout:
    each seed event -> its single nearest target event within +/- half_window
    (the pair), plus a CONTROL target from the same bout that is clearly not
    co-occurring (more than 2 x half_window away). Returns
    (seed_pos, target_pos, control_pos) tuples; control_pos is -1 if the bout
    has no such event."""
    out = []
    s_bouts = sg["nrem_bout_index"].values
    t_bouts = tg["nrem_bout_index"].values
    s_peaks, t_peaks = sg[peak_col].values, tg[peak_col].values
    for bout in np.unique(s_bouts):
        s_pos = np.where(s_bouts == bout)[0]
        t_pos = np.where(t_bouts == bout)[0]
        if len(s_pos) == 0 or len(t_pos) == 0:
            continue
        t_order = t_pos[np.argsort(t_peaks[t_pos])]
        t_sorted = t_peaks[t_order]
        for sp in s_pos:
            p = s_peaks[sp]
            j = np.searchsorted(t_sorted, p)
            cands = [k for k in (j - 1, j) if 0 <= k < len(t_sorted)]
            best = min(cands, key=lambda k: abs(t_sorted[k] - p))
            if abs(t_sorted[best] - p) > half_window:
                continue
            far = t_pos[np.abs(t_peaks[t_pos] - p) > 2 * half_window]
            cp = int(rng.choice(far)) if len(far) else -1
            out.append((sp, int(t_order[best]), cp))
    return out


def sampled_wave_similarity(
    seed_df, target_df, area_seed, area_target, wave, units, checkpoint_every=5
):
    key = f"{wave}_{area_seed}_{area_target}_frac{FILE_FRACTION}_seed{SAMPLE_SEED}_ctrl"
    final_f = CACHE_WAVE_SIM / f"{key}.pkl"
    ckpt_f = CACHE_WAVE_SIM / f"{key}_partial.pkl"
    if final_f.exists():
        print(f"  {area_seed}-{area_target} {wave}: loaded from cache")
        return pickle.load(open(final_f, "rb"))

    peak_col, half = f"{wave}_peak_index", WAVE_HALF_WINDOW_MS[wave]
    s_all = seed_df[seed_df["area"] == area_seed]
    t_all = target_df[target_df["area"] == area_target]

    rows, done = [], set()
    if ckpt_f.exists():
        ck = pickle.load(open(ckpt_f, "rb"))
        rows, done = ck["rows"], set(ck["done"])
        print(f"  resuming: {len(done)}/{len(units)} recordings already done")

    n_multi_channel, n_missing, n_retry_later, t0 = 0, 0, 0, time.time()
    for u_i, u in enumerate(units.itertuples(index=False)):
        ukey = (u.rat, str(u.date), u.trial)
        if ukey in done:
            continue
        m_s = (
            (s_all["rat"] == u.rat)
            & (s_all["date"] == u.date)
            & (s_all["trial"] == u.trial)
        )
        m_t = (
            (t_all["rat"] == u.rat)
            & (t_all["date"] == u.date)
            & (t_all["trial"] == u.trial)
        )
        sg, tg = s_all[m_s], t_all[m_t]

        unit_done = (
            True  # flipped to False only when the failure looks transient/retryable
        )

        if not sg.empty and not tg.empty:
            if sg["channel"].nunique() > 1 or tg["channel"].nunique() > 1:
                n_multi_channel += (
                    1  # ambiguous which trace -> skip, count it, done for good
                )
            else:
                rng = np.random.default_rng(
                    [
                        SAMPLE_SEED,
                        int(u.rat),
                        int(u.trial),
                        int(pd.Timestamp(u.date).strftime("%Y%m%d")),
                    ]
                )
                pairs = match_pairs_in_unit(sg, tg, peak_col, half, rng)
                if pairs:
                    try:
                        s_sig = filtered_signal(
                            u.rat,
                            u.date,
                            area_seed,
                            sg["channel"].iloc[0],
                            u.trial,
                            wave,
                        )
                        t_sig = filtered_signal(
                            u.rat,
                            u.date,
                            area_target,
                            tg["channel"].iloc[0],
                            u.trial,
                            wave,
                        )
                        s_st, s_en = (
                            sg[f"{wave}_start_index"].values,
                            sg[f"{wave}_end_index"].values,
                        )
                        t_st, t_en = (
                            tg[f"{wave}_start_index"].values,
                            tg[f"{wave}_end_index"].values,
                        )
                        for sp, tp, cp in pairs:
                            wa = s_sig[int(s_st[sp]) : int(s_en[sp])]
                            wb = t_sig[int(t_st[tp]) : int(t_en[tp])]
                            if len(wa) < 20 or len(wb) < 20:
                                continue
                            row = {
                                "rat": u.rat,
                                "date": u.date,
                                "trial": u.trial,
                                **wave_similarity(wa, wb),
                            }
                            wc = t_sig[int(t_st[cp]) : int(t_en[cp])] if cp >= 0 else []
                            if len(wc) >= 20:
                                c = wave_similarity(wa, wc)
                                row["ctrl_pearson_r"], row["ctrl_xcorr_peak"] = (
                                    c["pearson_r"],
                                    c["xcorr_peak"],
                                )
                            else:
                                row["ctrl_pearson_r"] = row["ctrl_xcorr_peak"] = np.nan
                            rows.append(row)
                    except UnreadableMatFile as e:
                        # out of retries (probably the drive), not marked done so it's retried next time
                        n_missing += 1
                        n_retry_later += 1
                        unit_done = False
                        print(
                            f"    skipping recording {ukey} (unreadable, will retry later): {e}"
                        )
                    except OSError as e:
                        # real FileNotFoundError (file doesn't exist), retrying won't help
                        n_missing += 1
                        print(
                            f"    skipping recording {ukey} (file not found, not retrying): {e}"
                        )

        _mat_cache.clear()
        _filtered_cache.clear()  # free RAM between recordings
        if unit_done:
            done.add(ukey)
        if len(done) % checkpoint_every == 0 and len(done) > 0:
            pickle.dump({"rows": rows, "done": list(done)}, open(ckpt_f, "wb"))
            print(
                f"    [checkpoint: {len(done)}/{len(units)} recordings done, "
                f"{n_retry_later} held back for retry, {len(rows):,} pairs, {time.time() - t0:.0f}s]"
            )

    still_pending = len(units) - len(done)
    if still_pending > 0:
        # some units still waiting for a retry, save as checkpoint
        pickle.dump({"rows": rows, "done": list(done)}, open(ckpt_f, "wb"))
        print(
            f"  {area_seed}-{area_target} {wave}: {still_pending} recording(s) still pending "
            f"retry (likely drive was unreachable) -- re-run this cell once the drive is back "
            f"to pick up where it left off. NOT writing a final cache yet."
        )
        return pd.DataFrame(rows)

    df = pd.DataFrame(rows)
    pickle.dump(df, open(final_f, "wb"))
    if ckpt_f.exists():
        ckpt_f.unlink()
    print(
        f"  {area_seed}-{area_target} {wave}: {len(df):,} pairs from "
        f"{df[['rat', 'date', 'trial']].drop_duplicates().shape[0] if len(df) else 0} recordings "
        f"(skipped: {n_multi_channel} multi-channel, {n_missing} missing/unreadable files)"
    )
    return df


units_by_wave = {"delta": sample_units(dlt_ctx), "spindle": sample_units(spn_ctx)}
for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
    n_all = ctx[["rat", "date", "trial"]].drop_duplicates().shape[0]
    u = units_by_wave[wave]
    print(
        f"{wave}: sampling {len(u)} of {n_all} recordings "
        f"({100 * len(u) / n_all:.1f}%) across {u['rat'].nunique()} rats"
    )

wave_sim_sampled_results = {}
for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
    for area_seed, area_target in PAIRS:
        print(f"=== {area_seed}-{area_target} {wave} ===")
        df = sampled_wave_similarity(
            ctx, ctx, area_seed, area_target, wave, units_by_wave[wave]
        )
        wave_sim_sampled_results[(area_seed, area_target, wave)] = df
        if not df.empty:
            d = df.dropna(subset=["ctrl_pearson_r"])
            per_rec = (
                d.assign(
                    dp=d["pearson_r"] - d["ctrl_pearson_r"],
                    dx=d["xcorr_peak"].abs() - d["ctrl_xcorr_peak"].abs(),
                )
                .groupby(["rat", "date", "trial"])[["dp", "dx"]]
                .median()
            )
            print(
                f"  pearson_r      paired {d['pearson_r'].median():+.3f} | control {d['ctrl_pearson_r'].median():+.3f}"
                f" | paired - control (median) {(d['pearson_r'] - d['ctrl_pearson_r']).median():+.3f}"
                f" | recordings with paired > control: {(per_rec['dp'] > 0).sum()}/{len(per_rec)}"
            )
            print(
                f"  |xcorr_peak|   paired {d['xcorr_peak'].abs().median():.3f} | control {d['ctrl_xcorr_peak'].abs().median():.3f}"
                f" | paired - control (median) {(d['xcorr_peak'].abs() - d['ctrl_xcorr_peak'].abs()).median():+.3f}"
                f" | recordings with paired > control: {(per_rec['dx'] > 0).sum()}/{len(per_rec)}"
            )

In [ ]:
wave_sim_summary_rows = []
for (area_seed, area_target, wave), df in wave_sim_sampled_results.items():
    if df.empty:
        continue
    d = df.dropna(subset=["ctrl_pearson_r"])
    if d.empty:
        continue
    per_rec = (
        d.assign(
            dp=d["pearson_r"] - d["ctrl_pearson_r"],
            dx=d["xcorr_peak"].abs() - d["ctrl_xcorr_peak"].abs(),
        )
        .groupby(["rat", "date", "trial"])[["dp", "dx"]]
        .median()
    )
    wave_sim_summary_rows.append(
        {
            "pair": f"{area_seed}-{area_target}",
            "wave": wave,
            "n_pairs": len(d),
            "n_recordings": len(per_rec),
            "pearson_paired": d["pearson_r"].median(),
            "pearson_control": d["ctrl_pearson_r"].median(),
            "pearson_diff": (d["pearson_r"] - d["ctrl_pearson_r"]).median(),
            "pearson_pct_above": 100 * (per_rec["dp"] > 0).mean(),
            "xcorr_paired": d["xcorr_peak"].abs().median(),
            "xcorr_control": d["ctrl_xcorr_peak"].abs().median(),
            "xcorr_diff": (d["xcorr_peak"].abs() - d["ctrl_xcorr_peak"].abs()).median(),
            "xcorr_pct_above": 100 * (per_rec["dx"] > 0).mean(),
        }
    )

wave_sim_summary = pd.DataFrame(wave_sim_summary_rows)
print(wave_sim_summary.round(3).to_string(index=False))
print(
    "How to read this: pearson_pct_above / xcorr_pct_above is the % of recordings where the"
)
print(
    "paired (co-occurring) waveforms were more similar than the control (non-co-occurring) pair."
)
print(
    "~50% means paired looks no different from chance; well above 50% is the pattern that would"
)
print("argue against volume conduction and toward a real, structured relationship.")
print(
    "Read pearson_diff/pearson_pct_above for delta, xcorr_diff/xcorr_pct_above for spindle --"
)
print(
    "Pearson at zero offset understates spindle, which has a real timing offset (~120 ms)."
)


if not wave_sim_summary.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    x = np.arange(len(wave_sim_summary))
    labels = [f"{r.pair}\n{r.wave}" for r in wave_sim_summary.itertuples()]

    for ax, col, title in [
        (axes[0], "pearson_pct_above", "Pearson r"),
        (axes[1], "xcorr_pct_above", "|xcorr_peak|"),
    ]:
        colors = [
            "tab:blue" if w == "delta" else "tab:orange"
            for w in wave_sim_summary["wave"]
        ]
        ax.bar(x, wave_sim_summary[col], color=colors, alpha=0.85)
        ax.axhline(50, color="k", lw=1.2, ls=":", label="chance (50%)")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylim(0, 100)
        ax.set_title(f"{title}: % recordings with paired > control")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, axis="y")

    fig.suptitle(
        "Wave similarity: paired vs. control, across all pairs and wave types",
        fontsize=12,
    )
    plt.tight_layout()
    save_fig(fig, "xregional_wavesim_summary")
    plt.show()

In [ ]:
print("delta units, full 8% sample:")
print(sample_units(dlt_ctx).head(10).to_string(index=False))
print(
    "total unique delta recordings available:",
    dlt_ctx[["rat", "date", "trial"]].drop_duplicates().shape[0],
)

In [ ]:
print("max delta_peak_index:", f"{dlt_ctx['delta_peak_index'].max():,.0f}")

## Does co-occurrence change over time?

Per-rat Spearman + permutation and plots only. y = ratio.

### NREM duration cutoff per day

Rat-days need at least `MIN_NREM_MIN` minutes of NREM for a per-day ratio (`n_coupled / n_seed`).

In [ ]:
MIN_NREM_MIN = 10  # minutes of NREM sleep required for a rat-day to count in the
# per-day trend and both HC-normalized versions only

bout_dur = dlt_ctx[BOUT_KEYS + ["nrem_bout_duration_s"]].drop_duplicates()
day_nrem = bout_dur.groupby(["rat", "date"])["nrem_bout_duration_s"].sum() / 60
good_days = day_nrem[day_nrem >= MIN_NREM_MIN].reset_index()[["rat", "date"]]
print(
    f"rat-days: {day_nrem.shape[0]} total -> {len(good_days)} with >= {MIN_NREM_MIN} min "
    f"NREM ({day_nrem.shape[0] - len(good_days)} dropped)"
)


def compute_cutoff_dropped(seed_df, target_df, area_seed, area_target, wave, window_ms):
    """Same per-day co-occurrence computation as per_day_cooccurrence, but
    restricted to the rat-days that FAILED the NREM cutoff (Step 9a)
    -- the ones per_day_cooccurrence silently drops. Returns their raw
    n_coupled, n_seed, ratio_proxy, and NREM minutes, so the cutoff's
    effect can be seen and shown to others rather than just asserted."""
    peak_col = f"{wave}_peak_index"
    lo, hi = window_ms
    s = seed_df[seed_df["area"] == area_seed]
    t = target_df[target_df["area"] == area_target]
    good_set = set(zip(good_days["rat"], good_days["date"]))
    rows = []
    for (rat, date), sg in s.groupby(["rat", "date"]):
        if (rat, date) in good_set:
            continue
        nrem_min = day_nrem.get((rat, date), float("nan"))
        if not (nrem_min < MIN_NREM_MIN):
            continue  # not dropped by the cutoff (e.g. 0 seed events), handled elsewhere
        tg = t[(t["rat"] == rat) & (t["date"] == date)]
        n_coupled = 0
        for trial in sg["trial"].unique():
            sg_t, tg_t = sg[sg["trial"] == trial], tg[tg["trial"] == trial]
            for bout in sg_t["nrem_bout_index"].unique():
                s_b = sg_t.loc[sg_t["nrem_bout_index"] == bout, peak_col].values
                t_b = (
                    tg_t.loc[tg_t["nrem_bout_index"] == bout, peak_col].values
                    if not tg_t.empty
                    else np.array([])
                )
                if len(s_b) == 0 or len(t_b) == 0:
                    continue
                lags = s_b[:, None] - t_b[None, :]
                n_coupled += int(((lags >= lo) & (lags < hi)).any(axis=1).sum())
        if len(sg) == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)]
        if row.empty:
            continue
        row = row.iloc[0]
        rows.append(
            {
                "rat": rat,
                "date": date,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
                "nrem_min": nrem_min,
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df["ratio_proxy"] = df["n_coupled"] / df["n_seed"].replace(0, np.nan)
    return df

In [ ]:
def per_day_cooccurrence(seed_df, target_df, area_seed, area_target, wave, window_ms):
    """Per rat-day: seed events with at least one target event at lag in
    [lo, hi). Counted within each trial and bout -- event indices restart every
    trial, so counting across trials (the earlier version) created spurious
    matches. n_seed is still the day's total seed events.

    Rat-days need >= MIN_NREM_MIN minutes of NREM sleep to be returned
    (good_days, computed above) -- replaces relying on raw event counts
    alone to judge whether a day's ratio is trustworthy."""
    peak_col = f"{wave}_peak_index"
    lo, hi = window_ms
    s = seed_df[seed_df["area"] == area_seed]
    t = target_df[target_df["area"] == area_target]
    rows = []
    for (rat, date), sg in s.groupby(["rat", "date"]):
        tg = t[(t["rat"] == rat) & (t["date"] == date)]
        n_coupled = 0
        for trial in sg["trial"].unique():
            sg_t, tg_t = sg[sg["trial"] == trial], tg[tg["trial"] == trial]
            for bout in sg_t["nrem_bout_index"].unique():
                s_b = sg_t.loc[sg_t["nrem_bout_index"] == bout, peak_col].values
                t_b = (
                    tg_t.loc[tg_t["nrem_bout_index"] == bout, peak_col].values
                    if not tg_t.empty
                    else np.array([])
                )
                if len(s_b) == 0 or len(t_b) == 0:
                    continue
                lags = s_b[:, None] - t_b[None, :]
                n_coupled += int(((lags >= lo) & (lags < hi)).any(axis=1).sum())
        if len(sg) == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)]
        if row.empty:
            continue
        row = row.iloc[0]
        rows.append(
            {
                "rat": rat,
                "date": date,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.merge(good_days, on=["rat", "date"])

In [ ]:
COHORTS = {
    "build-up (rats 1-8)": [r for r in RATS if r <= 8],
    "update (rats 9-16)": [r for r in RATS if r > 8],
}

In [ ]:
def find_missing_points(daily_df, rats_to_check, group="Training"):
    """Days expected (per rip_ctx) but ABSENT from daily_df entirely, with the
    REAL reason -- distinguishing a day that failed the NREM-duration
    cutoff (good_days / Step 9a) from a day that had zero seed-area events.
    Works for either group: Training days carry a GL/session label, HC days
    don't."""
    expected = rip_ctx[
        (rip_ctx["rat"].isin(rats_to_check)) & (rip_ctx["group"] == group)
    ]
    if group == "Training":
        expected = expected.dropna(subset=["gl_num", "session_num"])
        cols = ["rat", "date", "gl_num", "session_num"]
    else:
        cols = ["rat", "date"]
    expected = expected[cols].drop_duplicates()
    present = set(zip(daily_df["rat"], daily_df["date"]))
    good_set = set(zip(good_days["rat"], good_days["date"]))
    notes = []
    for _, row in expected.iterrows():
        key = (row["rat"], row["date"])
        if key in present:
            continue
        date_str = row["date"].date() if hasattr(row["date"], "date") else row["date"]
        if key not in good_set:
            nrem_min = day_nrem.get(key, float("nan"))
            reason = f"below NREM cutoff ({nrem_min:.1f} min < {MIN_NREM_MIN} min)"
        else:
            reason = "0 seed events that day"
        if group == "Training":
            notes.append(
                f"Rat {row['rat']}: {date_str} (GL{int(row['gl_num'])}_S{int(row['session_num'])}) "
                f"missing -- {reason}"
            )
        else:
            notes.append(f"Rat {row['rat']}: {date_str} (HC) missing -- {reason}")
    return notes


def spearman_permutation_plot(
    daily_df, area_seed, area_target, wave, ylim=None, dropped_df=None
):
    """Per-rat Spearman correlation on the ratio-to-chance proxy
    (n_coupled / n_seed), permutation p-value + z-score (1000 shuffles of
    day order). Cohort-split, GL-shaded x-axis. Footnotes (1) any training
    day missing from the plot entirely, with its real cause, and (2) any
    gap within a rat's own line caused by that rat simply never running a
    given session -- benign, not missing data. If dropped_df is given
    (cutoff-dropped Training rows from compute_cutoff_dropped), those raw
    points are drawn as hollow gray markers at their GL/session position --
    visible, but excluded from the trend/correlation."""
    from scipy.stats import spearmanr

    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(daily_df, all_rats_in_cohorts, group="Training")
    gap_notes = []

    sub_all = daily_df.copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all.dropna(subset=["gl_num", "session_num", "ratio_proxy"])
    if sub_all.empty:
        print(
            f"{area_seed}-{area_target} {wave}: no GL/session-labelled data, skipping"
        )
        return []

    drop_all = pd.DataFrame()
    if dropped_df is not None and not dropped_df.empty:
        drop_all = (
            dropped_df[dropped_df["group"] == "Training"]
            .dropna(subset=["gl_num", "session_num"])
            .copy()
        )

    rng = np.random.default_rng(0)
    results = []
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        dsub = (
            drop_all[drop_all["rat"].isin(rats)].copy()
            if not drop_all.empty
            else pd.DataFrame()
        )
        if sub.empty and dsub.empty:
            ax.set_title(label, fontsize=11)
            continue

        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        if not dsub.empty:
            dsub["step"] = dsub["gl_num"] * 10 + dsub["session_num"]
        order = sorted(
            set(sub["step"].unique())
            | (set(dsub["step"].unique()) if not dsub.empty else set())
        )
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                if len(r) > 0:
                    gap_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying day(s), need >=5"
                    )
                continue
            rho, _ = spearmanr(r["days_elapsed"], r["ratio_proxy"])
            perm_rhos = np.array(
                [
                    spearmanr(r["days_elapsed"], rng.permutation(r["ratio_proxy"]))[0]
                    for _ in range(1000)
                ]
            )
            p_perm = (np.sum(np.abs(perm_rhos) >= abs(rho)) + 1) / 1001
            z_perm = (rho - perm_rhos.mean()) / (perm_rhos.std() + 1e-9)

            results.append(
                {
                    "pair": f"{area_seed}-{area_target}",
                    "wave": wave,
                    "cohort": label,
                    "rat": rat,
                    "n_days": len(r),
                    "rho": rho,
                    "z_perm": z_perm,
                    "p_perm": p_perm,
                }
            )

            g = r.groupby("step")["ratio_proxy"].mean().reindex(order)
            xs = [pos[s] for s in order]
            ax.plot(
                xs,
                g.values,
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}  \u03c1={rho:+.2f}{'*' if p_perm < 0.05 else ''}",
            )

            missing_idx = sorted(pos[s] for s in order if s not in set(r["step"]))
            if missing_idx:
                ranges = []
                start = prev = missing_idx[0]
                for idx in missing_idx[1:]:
                    if idx == prev + 1:
                        prev = idx
                        continue
                    ranges.append((start, prev))
                    start = prev = idx
                ranges.append((start, prev))

                def _lab(s):
                    return f"GL{int(s // 10)}_S{int(s % 10)}"

                parts = [
                    _lab(order[a])
                    if a == b
                    else f"{_lab(order[a])}\u2013{_lab(order[b])}"
                    for a, b in ranges
                ]
                gap_notes.append(f"rat {rat}: no " + ", ".join(parts))

        if not dsub.empty:
            xs_d = [pos[s] for s in dsub["step"]]
            ax.scatter(
                xs_d,
                dsub["ratio_proxy"],
                marker="o",
                s=26,
                facecolors="none",
                edgecolors="0.5",
                linewidths=1.1,
                zorder=5,
                label="cutoff-dropped (raw)" if ax is axes[0] else None,
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")
        if ylim is not None:
            ax.set_ylim(ylim)

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  co-occurrence over time, Training days   "
        "(shading marks goal-location blocks; * = permutation p < 0.05; "
        "hollow circles = cutoff-dropped, shown raw)",
        fontsize=12,
    )

    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(gap_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if gap_notes:
            fig.text(
                0.5,
                y,
                "Line gap(s) -- rat did not run that session, not missing data: "
                + " | ".join(gap_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_overtime_{wave}_{area_seed}_{area_target}")
    plt.show()
    return results

### Raw ratio, Home-cage days

HC days have no GL/session label, so x = `days_elapsed`. Same y-axis as the training plot for that pair.

In [ ]:
def hc_overtime_plot(
    daily_df, area_seed, area_target, wave, ylim=None, dropped_df=None
):
    """Raw (non-normalized) ratio-to-chance proxy for HOME-CAGE days only,
    against days_elapsed. Footnotes any HC day missing from the plot with its
    real cause, and any rat excluded entirely for having fewer than 5
    qualifying HC days. If dropped_df is given (cutoff-dropped HC rows from
    compute_cutoff_dropped), those raw points are drawn as hollow gray
    markers at their days_elapsed position."""
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(daily_df, all_rats_in_cohorts, group="HC")
    excluded_notes = []

    sub_all = daily_df[daily_df["group"] == "HC"].copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all.dropna(subset=["ratio_proxy"])

    drop_all = pd.DataFrame()
    if dropped_df is not None and not dropped_df.empty:
        drop_all = dropped_df[dropped_df["group"] == "HC"].copy()

    if sub_all.empty and drop_all.empty:
        print(f"{area_seed}-{area_target} {wave}: no HC data, skipping")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        dsub = (
            drop_all[drop_all["rat"].isin(rats)].copy()
            if not drop_all.empty
            else pd.DataFrame()
        )
        if sub.empty and dsub.empty:
            ax.set_title(label, fontsize=11)
            continue
        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                if len(r) > 0:
                    excluded_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying HC day(s), need >=5"
                    )
                continue
            ax.plot(
                r["days_elapsed"],
                r["ratio_proxy"],
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}",
            )
        if not dsub.empty:
            ax.scatter(
                dsub["days_elapsed"],
                dsub["ratio_proxy"],
                marker="o",
                s=26,
                facecolors="none",
                edgecolors="0.5",
                linewidths=1.1,
                zorder=5,
                label="cutoff-dropped (raw)" if ax is axes[0] else None,
            )
        ax.set_title(label, fontsize=11)
        ax.set_xlabel("days elapsed")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")
        if ylim is not None:
            ax.set_ylim(ylim)

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  co-occurrence over time, Home-cage days (raw)   "
        "(hollow circles = cutoff-dropped, shown raw)",
        fontsize=12,
    )

    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(excluded_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if excluded_notes:
            fig.text(
                0.5,
                y,
                "Excluded rat(s) -- fewer than 5 qualifying HC days: "
                + " | ".join(excluded_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_hc_raw_{wave}_{area_seed}_{area_target}")
    plt.show()

In [ ]:
WINDOW_MS = (-750, 750)

xregional_overtime_results = []
all_cutoff_dropped = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        dropped_df = compute_cutoff_dropped(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        all_cutoff_dropped.append((area_seed, area_target, wave, dropped_df))

        if daily_df.empty:
            print(f"{area_seed}-{area_target} {wave}: no data, skipping")
            continue

        ratio_all = pd.concat(
            [
                daily_df["n_coupled"] / daily_df["n_seed"].replace(0, np.nan),
                dropped_df["ratio_proxy"]
                if not dropped_df.empty
                else pd.Series(dtype=float),
            ]
        ).dropna()
        shared_ylim = (0, ratio_all.max() * 1.08) if not ratio_all.empty else None

        xregional_overtime_results.extend(
            spearman_permutation_plot(
                daily_df,
                area_seed,
                area_target,
                wave,
                ylim=shared_ylim,
                dropped_df=dropped_df,
            )
        )
        hc_overtime_plot(
            daily_df,
            area_seed,
            area_target,
            wave,
            ylim=shared_ylim,
            dropped_df=dropped_df,
        )

In [ ]:
overtime_summary = pd.DataFrame(xregional_overtime_results)
overtime_summary["significant"] = overtime_summary["p_perm"] < 0.05

print("=== Step 9 -- per-rat Spearman correlation, co-occurrence ratio vs. time ===")
print("(Training days, raw/un-normalized ratio_proxy;")
print("* = permutation p < 0.05)")
print()
print(f"{len(overtime_summary)} rat-level permutation tests total\n")
print(
    overtime_summary.sort_values(["pair", "wave", "p_perm"])[
        [
            "pair",
            "wave",
            "cohort",
            "rat",
            "n_days",
            "rho",
            "z_perm",
            "p_perm",
            "significant",
        ]
    ].to_string(index=False)
)

print()
print("=== Step 9b -- Home-cage days, raw ratio_proxy ===")
print("Plotted per rat against calendar days_elapsed (HC days carry no GL/session")
print("label). Y-axis shared with the Training plot above for the same pair/wave,")
print("computed from the full (Training + HC + cutoff-dropped) ratio_proxy range,")
print("so it is not the same scale across different pairs. No correlation stats")
print("computed for HC -- purely descriptive, checking whether the HC baseline")
print("itself drifts.")
print()
print("Both plot types footnote their own missing points (NREM-cutoff day")
print("vs. 0-seed-events day); the Training plot additionally footnotes benign line")
print("gaps, and both show cutoff-dropped days as hollow gray markers at their raw")
print("value rather than silently excluding them.")
print()

print("=== Cutoff-dropped rat-days: raw ratio_proxy vs. NREM minutes ===")
print(f"(MIN_NREM_MIN = {MIN_NREM_MIN} -- every row below has less than this)")
print()
rows = []
for area_seed, area_target, wave, dropped_df in all_cutoff_dropped:
    if dropped_df.empty:
        continue
    d = dropped_df.copy()
    d["pair"] = f"{area_seed}-{area_target}"
    d["wave"] = wave
    rows.append(d)

if rows:
    cutoff_table = pd.concat(rows, ignore_index=True)
    cutoff_table["date"] = cutoff_table["date"].apply(
        lambda d: d.date() if hasattr(d, "date") else d
    )
    cutoff_table = cutoff_table.sort_values(["pair", "wave", "rat", "date"])
    print(f"{len(cutoff_table)} rat-day rows total, cut across all pairs/waves\n")
    print(
        cutoff_table[
            [
                "pair",
                "wave",
                "rat",
                "date",
                "group",
                "nrem_min",
                "n_coupled",
                "n_seed",
                "ratio_proxy",
            ]
        ].to_string(index=False)
    )
else:
    print("none -- no rat-day failed the NREM cutoff for any pair/wave")

# Over time plot normalised to the nearest HC session

In [ ]:
def nearest_hc_normalize(daily_df):
    """For each training day, divide that day's ratio_proxy (n_coupled / n_seed)
    by the nearest home-cage day's ratio_proxy. If the nearest HC day is 0,
    falls back to the second-nearest. If that's also 0, the training day is
    dropped (capped at two attempts) -- reported in dropped_points below."""
    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    dropped = []
    for rat, g in daily_df.groupby("rat"):
        hc = (
            g[g["group"] == "HC"]
            .dropna(subset=["ratio_proxy"])
            .sort_values("days_elapsed")
        )
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if hc.empty or train.empty:
            continue
        for _, row in train.iterrows():
            hc_sorted = hc.iloc[
                (hc["days_elapsed"] - row["days_elapsed"]).abs().argsort()
            ]
            used = None
            for rank, (_, hc_row) in enumerate(hc_sorted.iterrows()):
                if rank >= 2:
                    break
                if hc_row["ratio_proxy"] and hc_row["ratio_proxy"] > 0:
                    used = hc_row
                    break
            if used is None:
                nearest_two = hc_sorted.head(2)
                dropped.append(
                    {
                        "rat": rat,
                        "date": row["date"],
                        "days_elapsed": row["days_elapsed"],
                        "nearest_hc_days_elapsed": list(nearest_two["days_elapsed"]),
                    }
                )
                continue
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / used["ratio_proxy"],
                    "nearest_hc_days_elapsed": used["days_elapsed"],
                }
            )
    return pd.DataFrame(out_rows), dropped


def dropped_to_notes(dropped):
    notes = []
    for d in dropped:
        date_str = d["date"].date() if hasattr(d["date"], "date") else d["date"]
        notes.append(
            f"Rat {d['rat']}: {date_str} missing -- nearest two HC days "
            f"({d['nearest_hc_days_elapsed']}) both had 0 co-occurrences"
        )
    return notes


def plot_hc_normalized(norm_df, daily_df, dropped, area_seed, area_target, wave):
    """Same GL-shaded, per-cohort layout as spearman_permutation_plot, but
    plotting the nearest-HC-normalized ratio instead of the raw proportion --
    with a chance line at 1.0. Footnotes any training day missing entirely
    (with its real cause) plus any benign line gap, same as Step 9."""
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(
        daily_df, all_rats_in_cohorts, group="Training"
    ) + dropped_to_notes(dropped)
    gap_notes = []

    if norm_df.empty:
        print(f"{area_seed}-{area_target} {wave}: no data to normalize, skipping")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = norm_df[norm_df["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue
        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                if len(r) > 0:
                    gap_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying normalized day(s), need >=3"
                    )
                continue
            g = r.groupby("step")["ratio"].mean().reindex(order)
            ax.plot(
                [pos[s] for s in order],
                g.values,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )

            missing_idx = sorted(pos[s] for s in order if s not in set(r["step"]))
            if missing_idx:
                ranges = []
                start = prev = missing_idx[0]
                for idx in missing_idx[1:]:
                    if idx == prev + 1:
                        prev = idx
                        continue
                    ranges.append((start, prev))
                    start = prev = idx
                ranges.append((start, prev))

                def _lab(s):
                    return f"GL{int(s // 10)}_S{int(s % 10)}"

                parts = [
                    _lab(order[a])
                    if a == b
                    else f"{_lab(order[a])}\u2013{_lab(order[b])}"
                    for a, b in ranges
                ]
                gap_notes.append(f"rat {rat}: no " + ", ".join(parts))

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  normalized to nearest home-cage session   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(gap_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if gap_notes:
            fig.text(
                0.5,
                y,
                "Line gap(s) -- rat did not run that session, not missing data: "
                + " | ".join(gap_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_hcnorm_{wave}_{area_seed}_{area_target}")
    plt.show()


xregional_hcnorm_results = {}
all_dropped = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        if daily_df.empty:
            print(f"{area_seed}-{area_target} {wave}: no data, skipping")
            continue
        norm_df, dropped = nearest_hc_normalize(daily_df)
        xregional_hcnorm_results[(area_seed, area_target, wave)] = norm_df
        all_dropped.append((f"{area_seed}-{area_target} {wave}", dropped))
        plot_hc_normalized(norm_df, daily_df, dropped, area_seed, area_target, wave)

print()
print("=== Points removed: both nearest and second-nearest HC days were 0 ===")
any_dropped = False
for name, dropped in all_dropped:
    for d in dropped:
        any_dropped = True
        print(
            f"  [{name}] rat {d['rat']}, {d['date']} (days_elapsed={d['days_elapsed']}): "
            f"nearest two HC days ({d['nearest_hc_days_elapsed']}) both had 0 co-occurrences"
        )
if not any_dropped:
    print("  none -- every training day found a usable HC baseline within two tries")

## Over-time plots, normalized to average home-cage session

Training ratio / average of the rat's nonzero HC days. Missing points are listed in the footnote.

In [ ]:
def average_hc_normalize(daily_df):
    """For each training day, divide by the AVERAGE of that rat's usable
    (nonzero) HC-day ratio_proxy values. Zero-value HC days are excluded
    from the average (a 0-co-occurrence day isn't a trustworthy baseline
    and shouldn't pull the average down artificially)."""
    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    no_baseline = []
    for rat, g in daily_df.groupby("rat"):
        hc = g[g["group"] == "HC"].dropna(subset=["ratio_proxy"])
        hc_nonzero = hc[hc["ratio_proxy"] > 0]
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if train.empty:
            continue
        if hc_nonzero.empty:
            no_baseline.append({"rat": rat, "n_hc_days": len(hc), "n_hc_nonzero": 0})
            continue
        avg_hc = hc_nonzero["ratio_proxy"].mean()
        for _, row in train.iterrows():
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / avg_hc,
                    "avg_hc_value": avg_hc,
                    "n_hc_days_used": len(hc_nonzero),
                }
            )
    return pd.DataFrame(out_rows), no_baseline


def plot_avg_hc_normalized(
    norm_df, daily_df, no_baseline, area_seed, area_target, wave
):
    """Same GL-shaded, per-cohort layout as spearman_permutation_plot, but
    plotting the average-HC-normalized ratio instead of the raw proportion.
    Footnotes any training day missing entirely (with its real cause), any
    rat with no usable HC baseline at all, and any benign line gap."""
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    no_baseline_notes = [
        f"Rat {nb['rat']}: {nb['n_hc_days']} HC day(s), all 0 co-occurrences -- "
        f"no usable baseline, this rat excluded entirely"
        for nb in no_baseline
    ]
    missing_notes = (
        find_missing_points(daily_df, all_rats_in_cohorts, group="Training")
        + no_baseline_notes
    )
    gap_notes = []

    if norm_df.empty:
        print(f"{area_seed}-{area_target} {wave}: no data to normalize, skipping")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = norm_df[norm_df["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue
        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                if len(r) > 0:
                    gap_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying normalized day(s), need >=3"
                    )
                continue
            g = r.groupby("step")["ratio"].mean().reindex(order)
            ax.plot(
                [pos[s] for s in order],
                g.values,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )

            missing_idx = sorted(pos[s] for s in order if s not in set(r["step"]))
            if missing_idx:
                ranges = []
                start = prev = missing_idx[0]
                for idx in missing_idx[1:]:
                    if idx == prev + 1:
                        prev = idx
                        continue
                    ranges.append((start, prev))
                    start = prev = idx
                ranges.append((start, prev))

                def _lab(s):
                    return f"GL{int(s // 10)}_S{int(s % 10)}"

                parts = [
                    _lab(order[a])
                    if a == b
                    else f"{_lab(order[a])}\u2013{_lab(order[b])}"
                    for a, b in ranges
                ]
                gap_notes.append(f"rat {rat}: no " + ", ".join(parts))

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  normalized to average home-cage session   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )
    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(gap_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if gap_notes:
            fig.text(
                0.5,
                y,
                "Line gap(s) -- rat did not run that session, not missing data: "
                + " | ".join(gap_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_avghcnorm_{wave}_{area_seed}_{area_target}")
    plt.show()


xregional_avghcnorm_results = {}
all_no_baseline = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        if daily_df.empty:
            print(f"{area_seed}-{area_target} {wave}: no data, skipping")
            continue
        norm_df, no_baseline = average_hc_normalize(daily_df)
        xregional_avghcnorm_results[(area_seed, area_target, wave)] = norm_df
        all_no_baseline.append((f"{area_seed}-{area_target} {wave}", no_baseline))
        plot_avg_hc_normalized(
            norm_df, daily_df, no_baseline, area_seed, area_target, wave
        )

print()
print("=== Rats with NO usable HC baseline at all (every HC day was 0) ===")
any_missing = False
for name, no_baseline in all_no_baseline:
    for nb in no_baseline:
        any_missing = True
        print(
            f"  [{name}] rat {nb['rat']}: {nb['n_hc_days']} HC days total, "
            f"all had 0 co-occurrences -- no training days normalized for this rat"
        )
if not any_missing:
    print("  none -- every rat had at least one usable HC day")

## Check: trial matching (regression test for the fix)

Events are now matched within `(rat, date, trial)`; before, events from different trials on the same day could be matched.

Compares the notebook's matching with a separate trial-aware version. All change columns should be 0.

In [ ]:
from scipy.stats import spearmanr


def _peaks_by_group(ctx, area, peak_col):
    """{(rat, date, trial, bout): sorted peak indices} -- trial IS part of the key."""
    d = ctx[ctx["area"] == area]
    return {
        k: np.sort(g.values)
        for k, g in d.groupby(["rat", "date", "trial", "nrem_bout_index"])[peak_col]
    }


def nearest_diffs_by_trial(ctx, area_seed, area_target, wave, window_ms):
    """Same as nearest_within_window (cell above): each seed event -> its single
    nearest target event within +/- window_ms, giving seed - target. The only
    difference: matching happens WITHIN a trial, never across trials."""
    peak_col = f"{wave}_peak_index"
    S = _peaks_by_group(ctx, area_seed, peak_col)
    T = _peaks_by_group(ctx, area_target, peak_col)
    out = []
    for key, s in S.items():
        t = T.get(key)
        if t is None or len(t) == 0:
            continue
        j = np.searchsorted(t, s)
        d_lo = s - t[np.clip(j - 1, 0, len(t) - 1)]
        d_hi = s - t[np.clip(j, 0, len(t) - 1)]
        d = np.where(np.abs(d_hi) < np.abs(d_lo), d_hi, d_lo)
        out.append(d[np.abs(d) <= window_ms])
    return np.concatenate(out) if out else np.array([])


def ncoupled_by_trial(ctx, area_seed, area_target, wave, window_ms):
    """Same count as per_day_cooccurrence (Step 9): seed events with at least one
    target event at lag in [lo, hi), summed per (rat, date) -- but counted
    WITHIN each trial and bout, never across trials."""
    lo, hi = window_ms
    peak_col = f"{wave}_peak_index"
    S = _peaks_by_group(ctx, area_seed, peak_col)
    T = _peaks_by_group(ctx, area_target, peak_col)
    counts = {}
    for (rat, date, trial, bout), s in S.items():
        t = T.get((rat, date, trial, bout))
        if t is None or len(t) == 0:
            continue
        n = int(
            (
                (
                    np.searchsorted(t, s - lo, side="right")
                    - np.searchsorted(t, s - hi, side="right")
                )
                > 0
            ).sum()
        )
        counts[(rat, date)] = counts.get((rat, date), 0) + n
    return counts


# 1. do bout indices repeat across trials?
print(
    "1. How many (rat, date, bout) groups WOULD be mixed without trial in the grouping?:"
)
for name, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
    per_bout = ctx.groupby(["rat", "date", "nrem_bout_index"])["trial"].nunique()
    per_day = ctx.groupby(["rat", "date"])["trial"].nunique()
    print(
        f"{name:8}: {100 * (per_bout > 1).mean():5.1f}% of (rat, date, bout-index) groups contain "
        f"events from more than one trial | {100 * (per_day > 1).mean():5.1f}% of rat-days have >1 trial"
    )

# 2. histogram: current vs trial-aware matching
print(
    "2. Nearest-neighbor histograms: notebook function vs independent trial-aware version:"
)
_bw = globals().get("BIN_WIDTH", {"delta": 10, "spindle": 50})


def _peak(d, bw):
    h, e = np.histogram(d, bins=np.arange(-750, 750 + bw, bw))
    return e[np.argmax(h)] + bw / 2


rows = []
for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
    for a, b in PAIRS:
        cur = globals().get("xregional_diffs", {}).get((a, b, wave))
        if cur is None:
            cur = nearest_within_window(ctx, ctx, a, b, wave, 750)
        new = nearest_diffs_by_trial(ctx, a, b, wave, 750)
        rows.append(
            {
                "pair": f"{a}-{b}",
                "wave": wave,
                "n_current": len(cur),
                "n_trial_aware": len(new),
                "n_change_%": 100 * (len(new) - len(cur)) / max(len(cur), 1),
                "peak_current_ms": _peak(cur, _bw[wave]),
                "peak_trial_aware_ms": _peak(new, _bw[wave]),
                "median_current_ms": np.median(cur),
                "median_trial_aware_ms": np.median(new),
            }
        )
print(pd.DataFrame(rows).round(1).to_string(index=False))

# 3. per-day counts and per-rat trend: current vs trial-aware
print(
    "3. Step 9 (co-occurrence over time): notebook function vs independent trial-aware version:"
)
rows = []
for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
    for a, b in PAIRS:
        daily = per_day_cooccurrence(ctx, ctx, a, b, wave, WINDOW_MS)
        if daily.empty:
            continue
        new_n = ncoupled_by_trial(ctx, a, b, wave, WINDOW_MS)
        daily["n_trial"] = [
            new_n.get((r, d), 0) for r, d in zip(daily["rat"], daily["date"])
        ]
        changed = daily["n_trial"] != daily["n_coupled"]
        pct = (
            100
            * (daily["n_trial"] - daily["n_coupled"]).abs()
            / daily["n_coupled"].clip(lower=1)
        )
        s9 = daily.dropna(
            subset=["gl_num", "session_num"]
        )  # same rows as the over-time analysis
        d_rho = []
        for rat, r in s9.groupby("rat"):
            r = r.sort_values("days_elapsed")
            if len(r) < 5:
                continue
            x_cur = r["n_coupled"] / r["n_seed"].replace(0, np.nan)
            x_new = r["n_trial"] / r["n_seed"].replace(0, np.nan)
            ok = x_cur.notna() & x_new.notna()
            if ok.sum() >= 5 and x_cur[ok].nunique() > 1 and x_new[ok].nunique() > 1:
                d_rho.append(
                    abs(
                        spearmanr(r["days_elapsed"][ok], x_new[ok])[0]
                        - spearmanr(r["days_elapsed"][ok], x_cur[ok])[0]
                    )
                )
        rows.append(
            {
                "pair": f"{a}-{b}",
                "wave": wave,
                "rat_days": len(daily),
                "days_changed_%": 100 * changed.mean(),
                "median_change_%": pct.median(),
                "max_change_%": pct.max(),
                "max_|delta rho|": max(d_rho) if d_rho else np.nan,
            }
        )
print(pd.DataFrame(rows).round(3).to_string(index=False))

print("\nHow to read this:")
print(
    "  Section 1: the size of the bug that was fixed (share of groups that would have mixed trials)."
)
print("             It is expected to be large; it no longer affects anything.")
print(
    "  Sections 2 and 3: the notebook's functions now match within trial, so every change column"
)
print(
    "             (n_change_%, days_changed_%, median/max_change_%, max_|delta rho|) must be 0 and the"
)
print(
    "             peaks identical. Anything else = a regression: do not send the plots."
)

## Notes

- Nearest neighbour within +/-750 ms (Schreiner et al. 2022); the all-pairs circular shift is commented out.
- Wave similarity: 8% sample with control, cached in `wave_similarity_cache_files/`.
- Matching within (rat, date, trial) and bout. Plots made before this fix need to be redone; the last cell checks it.
- Windows: histograms and over-time counts +/-750 ms; wave-similarity pairs +/-200 ms (delta), +/-750 ms (spindle).
- Over-time analysis only uses rat-days with >= 10 min NREM (`MIN_NREM_MIN`).